In [1]:
from pathlib import Path
import importlib
import copy
import json
import tqdm

import os
import numpy as np
import pandas as pd
import torch

import matplotlib.pyplot as plt

from src.datasets import (
    prepare_data,
    split_dataset,
)
import os.path as osp

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# device = 'cpu'

In [4]:
exp_dir_path = Path("out_Termo_Ablation_heads/8_heads")
out_dir_path = osp.join(exp_dir_path, 'results')

In [5]:
with open(Path(exp_dir_path) / 'params.json', 'r') as f:
    cfg = json.load(f)
pair_dataset, pair_scalers , _, _, test_loader, pair_ideal = prepare_data(
    cfg['dataset'],
    cfg['dataloader'],
    cfg['utils']['seed']
)
cfg['dataset']['load'] = True

Датасет загружен из файла: data_Termo_Heat.pt
Готово! Количество графов: 42525, идеальных графов: 2
Train: 29767, Val: 6378, Test: 6380


In [6]:
train_dataset, val_dataset, test_dataset = split_dataset(
    pair_dataset,
    cfg['dataloader']['train_ratio'],
    cfg['dataloader']['val_ratio'],
    seed=cfg['utils']['seed']
)

In [7]:
print(len(pair_dataset))
print(pair_dataset[0])

42525
(Data(x=[107, 9], edge_index=[2, 107], edge_attr=[107, 5], global_attrs=[1, 5], edge_label=[1], edge_moded=[107, 1], nodes_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_nodes_pr2_fwd.csv', edges_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_tubes_pr2_fwd.csv'), Data(x=[107, 9], edge_index=[2, 107], edge_attr=[107, 5], global_attrs=[1, 5], edge_label=[1], edge_moded=[107, 1], nodes_fp='datasets/Termo_model_bwd/Tout_-5/problem/tube_119/Thermo_model_db_n_nodes_pr2_bwd.csv', edges_fp='datasets/Termo_model_bwd/Tout_-5/problem/tube_119/Thermo_model_db_n_tubes_pr2_bwd.csv'))


In [8]:
dataset= pair_dataset[0]
scalers = pair_scalers[0]
ideal_dataset = pair_ideal[0]

In [9]:
in_node_dim = dataset[0].x.shape[1]
in_edge_dim = dataset[0].edge_attr.shape[1]
model_module = importlib.import_module(f"src.models.{cfg['model']['name']}")
ModelClass = getattr(model_module, cfg['model']['name'])

def create_model():
    return ModelClass(
        in_node_dim=in_node_dim,
        in_edge_dim=in_edge_dim,
        **cfg['model']['kwargs']
    )

model = create_model().to(device)

state = torch.load(exp_dir_path / 'best_model.pth', weights_only=True)
model.load_state_dict(state)
model.eval()



EdgeClassifierNetwork_Attr(
  (node_encoder): NodeEncoder(
    (convs): ModuleList(
      (0): GATv2Conv(14, 128, heads=1)
      (1-7): 7 x GATv2Conv(128, 128, heads=1)
    )
    (norms): ModuleList(
      (0-7): 8 x LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (jump): JumpingKnowledge(cat)
  )
  (edge_init): Sequential(
    (0): Linear(in_features=2058, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
  )
  (edge_update_layers): ModuleList(
    (0-7): 8 x EdgeAttentionLayerFast(
      (q_proj): Linear(in_features=128, out_features=128, bias=True)
      (k_proj): Linear(in_features=128, out_features=128, bias=True)
      (v_proj): Linear(in_features=128, out_features=128, bias=True)
      (out_proj): Linear(in_features=128, out_features=128, bias=True)
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )


In [10]:
from torch_geometric.explain import Explainer, GNNExplainer, AttentionExplainer, GraphMaskExplainer, PGExplainer, CaptumExplainer
import torch_geometric

In [11]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x, edge_index, data):
        return self.model(x, edge_index, torch_geometric.data.Batch.from_data_list([data]))

In [12]:
wrapped = ModelWrapper(model)

In [13]:
dataset[0].to(device)

Data(x=[107, 9], edge_index=[2, 107], edge_attr=[107, 5], global_attrs=[1, 5], edge_label=[1], edge_moded=[107, 1], nodes_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_nodes_pr2_fwd.csv', edges_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_tubes_pr2_fwd.csv')

In [14]:
len(pair_dataset)

42525

In [15]:
def interpretation(model, dataset, fwd_or_bwd, node_mask_type = 'object', edge_mask_type = 'object'):
    '''
    Generate GNNExplainer explanations for forward (0) or backward (1) graph.
    
    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model for graph-level multiclass classification.
    dataset : dict
        Container with 'fwd' and 'bwd' graph data objects.
    fwd_or_bwd : int
        0 for forward graph, 1 for backward graph.
    node_mask_type : str or None, optional
        Type of node mask: None, 'object', 'attributes', or 'common_attributes'.
        Default 'object'.
    edge_mask_type : str or None, optional
        Type of edge mask: None or 'object'. Default 'object'.
    
    Returns
    -------
    explanation : Explanation object
        Contains node_mask and edge_mask with importance scores.
    '''
    
    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(),
        explanation_type='model',          # Объясняем предсказание модели
        node_mask_type=node_mask_type,       # Хотим понять важность признаков узлов
        edge_mask_type=edge_mask_type,           # Хотим понять важность ребер
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='log_probs',
        ),
    )
    dataset[fwd_or_bwd].to(device)
    explanation = explainer(dataset[fwd_or_bwd].x, dataset[fwd_or_bwd].edge_index, index=None, data=dataset[fwd_or_bwd])
    return explanation

In [16]:
explanation = interpretation(wrapped, pair_dataset[4242], 0, node_mask_type='object', edge_mask_type='object') 

In [17]:
print(explanation.node_mask)  # Важность узлов и их признаков
print(explanation.edge_mask)  # Важность ребер


tensor([[0.2922],
        [0.3030],
        [0.2709],
        [0.6889],
        [0.4937],
        [0.7252],
        [0.5186],
        [0.3734],
        [0.4765],
        [0.2917],
        [0.5762],
        [0.3965],
        [0.5277],
        [0.5727],
        [0.3262],
        [0.5334],
        [0.3518],
        [0.6749],
        [0.5265],
        [0.6899],
        [0.4908],
        [0.7010],
        [0.5069],
        [0.4717],
        [0.5553],
        [0.4275],
        [0.6331],
        [0.5510],
        [0.4027],
        [0.3990],
        [0.4270],
        [0.5083],
        [0.4180],
        [0.3737],
        [0.3651],
        [0.3181],
        [0.4259],
        [0.5881],
        [0.6599],
        [0.4163],
        [0.5902],
        [0.4979],
        [0.5477],
        [0.5794],
        [0.5092],
        [0.5288],
        [0.5328],
        [0.5893],
        [0.6075],
        [0.5066],
        [0.4765],
        [0.7674],
        [0.7331],
        [0.3087],
        [0.5498],
        [0

In [18]:
explanation = interpretation(wrapped, test_dataset[42], 0, node_mask_type='object', edge_mask_type='object') 

In [19]:
print(explanation.node_mask)
print(explanation.edge_mask)

tensor([[0.3521],
        [0.3074],
        [0.3385],
        [0.6830],
        [0.3549],
        [0.6286],
        [0.6525],
        [0.4710],
        [0.4885],
        [0.6207],
        [0.4441],
        [0.5967],
        [0.5332],
        [0.2667],
        [0.2922],
        [0.2795],
        [0.2636],
        [0.3129],
        [0.3367],
        [0.3646],
        [0.3191],
        [0.5467],
        [0.3201],
        [0.2458],
        [0.2857],
        [0.2738],
        [0.3014],
        [0.3457],
        [0.5215],
        [0.5628],
        [0.5391],
        [0.2829],
        [0.4981],
        [0.4979],
        [0.5951],
        [0.6453],
        [0.2590],
        [0.6748],
        [0.6502],
        [0.2874],
        [0.3341],
        [0.4668],
        [0.5298],
        [0.3198],
        [0.5340],
        [0.3689],
        [0.2965],
        [0.3447],
        [0.3281],
        [0.3116],
        [0.7006],
        [0.4506],
        [0.5740],
        [0.5889],
        [0.3689],
        [0

In [20]:
batch = next(iter(test_loader))

In [21]:
batch

[DataBatch(x=[1712, 9], edge_index=[2, 1712], edge_attr=[1712, 5], global_attrs=[16, 5], edge_label=[16], edge_moded=[1712, 1], nodes_fp=[16], edges_fp=[16], batch=[1712], ptr=[17]),
 DataBatch(x=[1712, 9], edge_index=[2, 1712], edge_attr=[1712, 5], global_attrs=[16, 5], edge_label=[16], edge_moded=[1712, 1], nodes_fp=[16], edges_fp=[16], batch=[1712], ptr=[17])]

In [22]:
explanation = interpretation(wrapped, batch, 0, node_mask_type='common_attributes', edge_mask_type='object') # Можно и батчи целые подавать (ОДНАКО ТАМ БУДУТ ПОПАДАТЬСЯ РАЗНЫЕ КЛАССЫ)

In [23]:
node_attr = ['pos_x', 'pos_y', 'types_def', 'types_usr', 'types_src', 'P', 'Temp', 'P_ideal', 'Temp_ideal']  # Атрибуты узлов
print(explanation.node_mask)  # Важность узлов и их признаков

tensor([[0.4262, 0.5926, 0.3522, 0.3482, 0.0000, 0.4459, 0.4864, 0.4664, 0.5102]],
       device='cuda:0')


In [24]:
print(explanation.edge_mask)  # Важность ребер


tensor([0.2813, 0.2718, 0.2656,  ..., 0.2591, 0.2719, 0.2793], device='cuda:0')


In [25]:
edge_attr = ['d', 'l', 'Vid_fwd', 'Vid_bwd', 'Vid_usr']  # Атрибуты ребер